In [ ]:
!pip install transformers torch pandas numpy

In [ ]:
import re

with open("ocr_result_clean.txt", "r", encoding="utf-8") as f:
    clean_text = f.read()

with open("ocr_result_degraded.txt", "r", encoding="utf-8") as f:
    noisy_text = f.read()

print("Чистий текст:")
print(clean_text[:500])

print("\nЗашумлений текст:")
print(noisy_text[:500])

date_pattern = r"\b\d{2}[-./]\d{2}[-./]\d{2,4}\b"

time_pattern = r"\b\d{2}:\d{2}:\d{2}\b"

datetime_pattern = (
    r"\b\d{2}[-./]\d{2}[-./]\d{2,4}"
    r"[\sT]*"
    r"\d{2}[:.,]\d{2}[:.,]\d{2}\b"
)

def analyze_text(text, title):
    print(f"\n{'=' * 50}")
    print(title)

    dates = re.findall(date_pattern, text)
    times = re.findall(time_pattern, text)
    datetimes = re.findall(datetime_pattern, text)

    print("Дати:", dates)
    print("Час:", times)
    print("Дата + час (покращений RegEx):", datetimes)

analyze_text(clean_text, "ЧИСТИЙ ТЕКСТ")
analyze_text(noisy_text, "ЗАШУМЛЕНИЙ ТЕКСТ")

Чистий текст:
Тзов
ТВК
'Львівхолод
УКРАІНА
ЛЬВІВСЬКА
ОБЛАСТЬ
РАДЕХІВСЬКНЙ
РАЙОН/М РАДЕХІВ ₽-H,
CNT . ЛОПАТИН
ВУЛ €
СТРІЛЬЦІВ
МАГАЗИН
ПН:
015536813049
01553681
Оператор
Гуненюк
Валентина
Михайлівна
Чек
246
588327
2854196
[2335]
Касса:
[z]
31z
ПФ ГОНІЛКА
КУРЯЧА 1КГ
КУРЯТ_ОХО
98.27
КЕТЧУП ДО
ШЕШЛИА 
дой/п 4о9г
TОР
ЧИН
ШАКПІН МАРИН ЦІЛІ
420Г
RIO
51.24
000
CKP
GOUDA 403
тоГ
MOLENDAM
АНАНАСИ КУСОЧ 58ОГ
RIO
51.74
000
ПРИПР
КАРТОПЛІ
зог ЛЮБИСТОК
31.32
ГОТІВКА
491
ГРН
РЕШТА
120
280
ПДВ
00%
Заокруглення:
сплати:
281
АР

Зашумлений текст:
таов
твт
Львівколод"
УKP-iHF ;
Львівськн
овл-с71
АдЕХІвСЬкин
РPYOH/N.₽AEXIB
сKк . ЛОПАтиН
ВУЛ.€ СТРІЛьШІв
ААГАз]
тн:
815525813849
Iд: 01558681
Оператср:
"пАетт+
Валентина Мнхайлівна
цet
246
{завз27
7854196
[2357
Kaсса:
11
з12
74.98
пe -оwIн
#РяЧА 1г
IFFT _ю5
9в.27
кЕЧ
4 шАшмку
рW/п 4885 ТОP
чд
19.85
шхпІн ААРиH цилі
4257
FIS
51.24
553
СYP
соu5-
48+ 79г
KOLERD-Y
АНАнАсК КУСОЧ 5в8г
Rzo
51.7-
888
18.4-
ПРУПЕ
до щнтСПЛІ з91
лв"стот
31 2
тотівк -
481.88 -РH
вешF
175,

In [ ]:

import re

test_cases = [
    "18.11.2021 14:22:52",
    "18.11.202114:22:52",
    "18.11.2021 14.22.52",
    "18.11.2021 14,22,52",
    "18-11-2021 14:22:52",
    "18/11/2021 14:22:52"
]

pattern = (
    r"\b\d{2}[-./]\d{2}[-./]\d{4}"
    r"[\sT]*"
    r"\d{2}[:.,]\d{2}[:.,]\d{2}\b"
)

print("ТЕСТУВАННЯ ПОКРАЩЕНОГО REGEX\n")

for text in test_cases:
    match = re.search(pattern, text)

    if match:
        print(f"[OK] {text}")
        print(f"     Знайдено: {match.group()}")
    else:
        print(f"[ERROR] {text}")


ТЕСТУВАННЯ ПОКРАЩЕНОГО REGEX

[OK] 18.11.2021 14:22:52
     Знайдено: 18.11.2021 14:22:52
[OK] 18.11.202114:22:52
     Знайдено: 18.11.202114:22:52
[OK] 18.11.2021 14.22.52
     Знайдено: 18.11.2021 14.22.52
[OK] 18.11.2021 14,22,52
     Знайдено: 18.11.2021 14,22,52
[OK] 18-11-2021 14:22:52
     Знайдено: 18-11-2021 14:22:52
[OK] 18/11/2021 14:22:52
     Знайдено: 18/11/2021 14:22:52


In [ ]:
import torch
from transformers import pipeline

models = {
    "BERT": "dslim/bert-base-NER",
    "WikiNEuRal": "Babelscape/wikineural-multilingual-ner",
    "DistilBERT": "elastic/distilbert-base-cased-finetuned-conll03-english"
}

device = 0 if torch.cuda.is_available() else -1

clean_sample = clean_text[:350]
noisy_sample = noisy_text[:350]

ner_results = {}

for model_name, model_id in models.items():
    print(f"\n{'=' * 50}")
    print(f"Модель: {model_name}")

    try:
        ner = pipeline(
            "ner",
            model=model_id,
            tokenizer=model_id,
            aggregation_strategy="simple",
            device=device
        )

        clean_entities = ner(clean_sample)
        noisy_entities = ner(noisy_sample)

        ner_results[model_name] = {
            "clean": clean_entities,
            "noisy": noisy_entities
        }

        print("\nЧИСТИЙ ТЕКСТ:")
        for entity in clean_entities:
            print(
                entity["word"],
                "|",
                entity["entity_group"],
                "|",
                round(entity["score"], 3)
            )

        print("\nЗАШУМЛЕНИЙ ТЕКСТ:")
        for entity in noisy_entities:
            print(
                entity["word"],
                "|",
                entity["entity_group"],
                "|",
                round(entity["score"], 3)
            )

    except Exception as error:
        print(f"Помилка моделі {model_name}: {error}")


Модель: BERT


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (18 outputs left out)

tokenizer_config.json:   0%|          | 0.00/257 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


ЧИСТИЙ ТЕКСТ:
##п | PER | 0.741
##е | PER | 0.775
##р | PER | 0.746
##а | PER | 0.823
##т | PER | 0.802
##о | PER | 0.818
##р Гуненюк | PER | 0.561
##и | PER | 0.306
##й | PER | 0.258
##в | LOC | 0.287
##а | LOC | 0.389
##с | LOC | 0.551
##с | LOC | 0.531
##а | LOC | 0.456

ЗАШУМЛЕНИЙ ТЕКСТ:
У | ORG | 0.846
##K | ORG | 0.896
##P | ORG | 0.918
i | ORG | 0.617
##HF | ORG | 0.704
Л | PER | 0.63
##ь | PER | 0.55
##в | PER | 0.65
##і | PER | 0.625
##в | PER | 0.464
##с | PER | 0.393
А | LOC | 0.387
О | PER | 0.532
##п | PER | 0.709
##е | PER | 0.596
В | PER | 0.394
##а | PER | 0.583
##л | PER | 0.51
##е | PER | 0.593
##н | PER | 0.675
##т | PER | 0.603
##и | PER | 0.656
##на | PER | 0.427
##нхайлі | LOC | 0.443
##п | PER | 0.522
##І | PER | 0.553


In [ ]:
import json
import pandas as pd

summary = []
json_output = {}

for model_name, results in ner_results.items():
    clean_entities = results["clean"]
    noisy_entities = results["noisy"]

    clean_loc = [
        e for e in clean_entities
        if e["entity_group"] in ["LOC", "B-LOC", "I-LOC"]
    ]

    noisy_loc = [
        e for e in noisy_entities
        if e["entity_group"] in ["LOC", "B-LOC", "I-LOC"]
    ]

    clean_count = len(clean_loc)
    noisy_count = len(noisy_loc)

    if clean_count > 0:
        change_percent = (
            (noisy_count - clean_count) / clean_count
        ) * 100
    else:
        change_percent = None

    summary.append({
        "Модель": model_name,
        "LOC (чистий)": clean_count,
        "LOC (шум)": noisy_count,
        "Зміна (%)": (
            round(change_percent, 2)
            if change_percent is not None else "Н/Д"
        )
    })

    json_output[model_name] = {
        "clean": [
            {
                "text": e["word"],
                "type": e["entity_group"],
                "confidence": round(float(e["score"]), 4)
            }
            for e in clean_entities
        ],
        "degraded": [
            {
                "text": e["word"],
                "type": e["entity_group"],
                "confidence": round(float(e["score"]), 4)
            }
            for e in noisy_entities
        ],
        "loc_clean_count": clean_count,
        "loc_degraded_count": noisy_count,
        "loc_change_percent": change_percent
    }

df_summary = pd.DataFrame(summary)

print("ПОРІВНЯННЯ МОДЕЛЕЙ NER")
print(df_summary.to_string(index=False))

with open("ner_extracted.json", "w", encoding="utf-8") as f:
    json.dump(json_output, f, ensure_ascii=False, indent=4)

print("\nСтворено файл ner_extracted.json")

NameError: name 'ner_results' is not defined

In [ ]:
import pandas as pd

def normalize_entity(text):
    return " ".join(text.casefold().split())

rows = []

for model_name, results in ner_results.items():
    clean = results["clean"]
    noisy = results["noisy"]

    clean_map = {
        (normalize_entity(e["word"]), e["entity_group"]): e
        for e in clean
    }

    noisy_map = {
        (normalize_entity(e["word"]), e["entity_group"]): e
        for e in noisy
    }

    matched = clean_map.keys() & noisy_map.keys()

    pr = (
        len(matched) / len(clean_map) * 100
        if clean_map else None
    )

    delta = (
        sum(
            float(clean_map[k]["score"]) -
            float(noisy_map[k]["score"])
            for k in matched
        ) / len(matched)
        if matched else None
    )

    clean_classes = {
        normalize_entity(e["word"]): e["entity_group"]
        for e in clean
    }

    noisy_classes = {
        normalize_entity(e["word"]): e["entity_group"]
        for e in noisy
    }

    drift = sum(
        clean_classes[t] != noisy_classes[t]
        for t in clean_classes.keys() & noisy_classes.keys()
    )

    rows.append({
        "Модель": model_name,
        "PR (%)": round(pr, 2) if pr is not None else "Н/Д",
        "Confidence Delta": (
            round(delta, 4) if delta is not None else "Н/Д"
        ),
        "Class Drift": drift
    })

metrics_df = pd.DataFrame(rows)

print("МЕТРИКИ СТІЙКОСТІ NER")
print(metrics_df.to_string(index=False))

NameError: name 'ner_results' is not defined

In [ ]:
from transformers import pipeline

model_id = "Babelscape/wikineural-multilingual-ner"

ner_tokens = pipeline(
    "ner",
    model=model_id,
    aggregation_strategy="none",
    device=device
)

sample = clean_text[:200]

print("ТОКЕНІЗАЦІЯ ТА BIO-МІТКИ\n")

entities = ner_tokens(sample)

for entity in entities:
    print(
        f"Токен: {entity['word']:<20} "
        f"Мітка: {entity['entity']:<10} "
        f"Confidence: {entity['score']:.4f}"
    )

print("\nWORDPIECE ТОКЕНИ:")
print(ner_tokens.tokenizer.tokenize(
    "Львівська область Радехів Лопатин"
))

NameError: name 'device' is not defined

In [ ]:
import json

model_name = "Babelscape/wikineural-multilingual-ner"

entities = ner_results["WikiNEuRal"]["clean"]

output = {
    "document_metadata": {
        "source_file": "ocr_result_clean.txt",
        "model_name": model_name,
        "aggregation_strategy": "simple",
        "total_entities_found": len(entities)
    },
    "entities": [
        {
            "entity_group": e["entity_group"],
            "word": e["word"],
            "score": round(float(e["score"]), 4),
            "start": int(e["start"]),
            "end": int(e["end"])
        }
        for e in entities
    ]
}

with open("ner_extracted.json", "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=4)

print(json.dumps(output, ensure_ascii=False, indent=4))
print("\nФайл ner_extracted.json збережено!")

NameError: name 'ner_results' is not defined

In [ ]:
import re
import torch
import pandas as pd
from google.colab import files
from transformers import pipeline

uploaded = files.upload()

with open("ocr_result_clean.txt", encoding="utf-8") as f:
    clean_text = f.read()

with open("ocr_result_degraded.txt", encoding="utf-8") as f:
    degraded_text = f.read()

model_id = "Babelscape/wikineural-multilingual-ner"

ner = pipeline(
    "ner",
    model=model_id,
    tokenizer=model_id,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

clean_entities = ner(clean_text[:350])
degraded_entities = ner(degraded_text[:350])

def show_entities(title, entities):
    print(f"\n=== {title} ===")
    for e in entities:
        print(
            f"{e['word']:<30} "
            f"{e['entity_group']:<8} "
            f"Score: {e['score']:.4f}"
        )

show_entities("CLEAN", clean_entities)
show_entities("DEGRADED", degraded_entities)

def normalize(text):
    return " ".join(text.casefold().split())

clean_map = {
    (normalize(e["word"]), e["entity_group"]): e
    for e in clean_entities
}

degraded_map = {
    (normalize(e["word"]), e["entity_group"]): e
    for e in degraded_entities
}

matched = clean_map.keys() & degraded_map.keys()

pr = (
    len(matched) / len(clean_map) * 100
    if clean_map else None
)

delta = (
    sum(
        float(clean_map[k]["score"]) -
        float(degraded_map[k]["score"])
        for k in matched
    ) / len(matched)
    if matched else None
)

clean_classes = {
    normalize(e["word"]): e["entity_group"]
    for e in clean_entities
}
degraded_classes = {
    normalize(e["word"]): e["entity_group"]
    for e in degraded_entities
}

common = clean_classes.keys() & degraded_classes.keys()

class_drift = sum(
    clean_classes[t] != degraded_classes[t]
    for t in common
)

print("\n=== МЕТРИКИ NER ===")
print("Clean entities:", len(clean_entities))
print("Degraded entities:", len(degraded_entities))
print("Matched entities:", len(matched))
print("Preservation Rate:", pr)
print("Delta Score:", delta)
print("Class Drift:", class_drift)

tokenizer = ner.tokenizer

words = [
    "Львівхолод",
    "Львівколод"
]

print("\n=== WORDPIECE ===")

for word in words:
    tokens = tokenizer.tokenize(word)
    print(f"{word} -> {tokens}")

df = pd.DataFrame([{
    "Model": "WikiNEuRal",
    "Clean entities": len(clean_entities),
    "Degraded entities": len(degraded_entities),
    "PR (%)": pr,
    "Delta Score": delta,
    "Class Drift": class_drift
}])

df.to_csv(
    "colloquium_bert_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nФайл colloquium_bert_metrics.csv створено.")


Saving ocr_result_clean.txt to ocr_result_clean (1).txt
Saving ocr_result_degraded.txt to ocr_result_degraded (1).txt


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  709MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/333 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


=== CLEAN ===
Гуненюк Валентина Михайлівна Чек PER      Score: 0.9623

=== DEGRADED ===
Валентина Мнхайлівна           PER      Score: 0.9954

=== МЕТРИКИ NER ===
Clean entities: 1
Degraded entities: 1
Matched entities: 0
Preservation Rate: 0.0
Delta Score: None
Class Drift: 0

=== WORDPIECE ===
Львівхолод -> ['Львів', '##хо', '##лод']
Львівколод -> ['Львів', '##кол', '##од']

Файл colloquium_bert_metrics.csv створено.


In [ ]:
def analyze_full_text(text, chunk_size=300):
    entities = []

    for start in range(0, len(text), chunk_size):
        chunk = text[start:start + chunk_size]
        result = ner(chunk)

        for entity in result:
            entities.append(entity)

    return entities


clean_entities = analyze_full_text(clean_text)
degraded_entities = analyze_full_text(degraded_text)

show_entities("CLEAN FULL TEXT", clean_entities)
show_entities("DEGRADED FULL TEXT", degraded_entities)

print("\nКількість сутностей:")
print("Clean:", len(clean_entities))
print("Degraded:", len(degraded_entities))


=== CLEAN FULL TEXT ===
Гуненюк Валентина Михайлівна Чек PER      Score: 0.9615

=== DEGRADED FULL TEXT ===
Валентина Мнхайлівна           PER      Score: 0.9950

Кількість сутностей:
Clean: 1
Degraded: 1


In [ ]:
tokenizer = ner.tokenizer

words = [
    "Львівхолод",
    "Львівколод",
    "Михайлівна",
    "Мнхайлівна"
]

print("=== WORDPIECE ===")

for word in words:
    tokens = tokenizer.tokenize(word)
    print(f"{word} -> {tokens}")

=== WORDPIECE ===
Львівхолод -> ['Львів', '##хо', '##лод']
Львівколод -> ['Львів', '##кол', '##од']
Михайлівна -> ['Ми', '##хай', '##лів', '##на']
Мнхайлівна -> ['Мн', '##хай', '##лів', '##на']
